# How to use the Compatibility API

The `/compatibility` endpoint helps you understand how to call TiTiler-CMR for a collection. It returns the sampled `granule_ur` and accepts `granule_ur` when you want to inspect a specific granule. For hierarchical HDF5 and NetCDF datasets it can return a lightweight list of candidate `group` paths from the root request, then let you inspect variables by making a follow-up request with `group=...`.

This notebook demonstrates that workflow for the [NISAR Beta Geocoded Polarimetric Covariance (GCOV)](https://www.earthdata.nasa.gov/data/catalog/asf-nisar-l2-gcov-beta-v1-1) collection, which is a known example of a hierarchical HDF5 dataset that needs a `group` parameter for xarray requests.

## Setup

In [ ]:
import json
import os

import earthaccess
import httpx2 as httpx

titiler_endpoint = os.getenv(
    "TITILER_CMR_ENDPOINT", "https://staging.openveda.cloud/api/titiler-cmr"
)

## Identify the dataset

You can find the collection with `earthaccess.search_datasets`.

In [ ]:
datasets = earthaccess.search_datasets(concept_id="C3622214170-ASF")
ds = datasets[0]

collection_concept_id = ds["meta"]["concept-id"]
print("CollectionConcept-Id:", collection_concept_id)
print("Short name:", ds["umm"]["ShortName"])
print("Version:", ds["umm"]["Version"])

## Explore the collection using `/compatibility`

In [ ]:
compatibility_response = httpx.get(
    f"{titiler_endpoint}/compatibility",
    params={"collection_concept_id": collection_concept_id},
    timeout=None,
).json()

print(json.dumps(compatibility_response, indent=2))

## Interpret the response

For hierarchical datasets, a root `/compatibility` request stays lightweight. The most important fields are:

- `granule_ur`: the sampled granule used for this compatibility check
- `compatible_groups`: candidate group paths to try in a follow-up request

At this stage the response may not include variable metadata or xarray links yet, because TiTiler-CMR has not inspected a nested group.

In [ ]:
compatible_groups = compatibility_response.get("compatible_groups", [])
variables = list((compatibility_response.get("variables") or {}).keys())
tilejson_link = next(
    (
        link["href"]
        for link in compatibility_response.get("links", [])
        if link["rel"] == "tilejson"
    ),
    None,
)

print("compatible_groups:")
for group in compatible_groups:
    print(" -", group)

## Inspect a specific group

Once you choose a group from `compatible_groups`, make a follow-up `/compatibility` request with `group=...` to inspect variables and get xarray links.

In [ ]:
selected_group = compatible_groups[1]

grouped_compatibility_response = httpx.get(
    f"{titiler_endpoint}/compatibility",
    params={
        "collection_concept_id": collection_concept_id,
        "group": selected_group,
    },
    timeout=30,
).json()

print(json.dumps(grouped_compatibility_response, indent=2))

In [ ]:
group_variables = list((grouped_compatibility_response.get("variables") or {}).keys())
group_tilejson_link = next(
    link["href"]
    for link in grouped_compatibility_response.get("links", [])
    if link["rel"] == "tilejson"
)

print("selected group:", selected_group)
print("variables in selected group (first 10):", group_variables[:10])
print("tilejson link for selected group:", group_tilejson_link)

For a grouped dataset like NISAR GCOV, the workflow is:

1. Call `/compatibility` without `group` to get candidate `compatible_groups`.
2. Choose one of those group paths.
3. Call `/compatibility` again with `group=...` to inspect variables and dimensions for that group.
4. Pick a variable name from `variables`.
5. Reuse the returned xarray links from the grouped response.

That keeps the root compatibility response lightweight while still giving you a practical path to the next xarray request.